# Manage fine-tuning sessions

Interactive companion to [`docs/session-management.md`](../docs/session-management.md). Use this notebook to:

- list the fine-tuning sessions your project has accumulated,
- inspect a single session (status, base model, LoRA config),
- list and inspect its checkpoints.

It is a thin shell over `client.sessions` and `client.checkpoints` in the `azure-ai-finetuningsessions` SDK — the same calls work against any project that has the preview feature enabled.

> **Prerequisites.** You've installed the cookbook (`pip install -e . --index-url https://pypi.org/simple`) and authenticated as described in [`docs/auth.md`](../docs/auth.md).

## 1. Configure the project endpoint

Set `AZURE_AI_PROJECT_ENDPOINT` before launching the notebook. The older `FOUNDRY_PROJECT_ENDPOINT` name is also accepted. Use the Foundry project URL, not a resource root URL; keep API keys out of notebook source and saved outputs.

In [ ]:
import os
from urllib.parse import urlparse

PROJECT_ENDPOINT = (
    os.environ.get("AZURE_AI_PROJECT_ENDPOINT")
    or os.environ.get("FOUNDRY_PROJECT_ENDPOINT")
    or ""
).strip()
endpoint_url = urlparse(PROJECT_ENDPOINT)
if endpoint_url.scheme != "https" or not endpoint_url.hostname or "<" in PROJECT_ENDPOINT:
    raise ValueError(
        "Set AZURE_AI_PROJECT_ENDPOINT to your Foundry project URL, for example "
        "https://<your-project>.services.ai.azure.com/api/projects/<project-name>. "
        "See docs/auth.md."
    )
print("endpoint host:", endpoint_url.hostname)

## 2. Build the client

Picks `AzureKeyCredential` if `AZURE_AI_API_KEY` is set, otherwise falls back to `DefaultAzureCredential` (i.e. your `az login` session). This mirrors what the math_rl recipe does.

In [ ]:
from azure.ai.finetuningsessions import FineTuningSessionClient
from azure.ai.finetuningsessions.models import FoundryFeaturesOptInKeys

PREVIEW = FoundryFeaturesOptInKeys.FINETUNING_SESSIONS_V1_PREVIEW
API_VERSION = "v1"

api_key = os.environ.get("AZURE_AI_API_KEY")
if api_key:
    from azure.core.credentials import AzureKeyCredential
    credential = AzureKeyCredential(api_key)
    extra: dict = {}
    print("auth: AzureKeyCredential")
else:
    from azure.identity import DefaultAzureCredential
    credential = DefaultAzureCredential(exclude_managed_identity_credential=True)
    extra = {"credential_scopes": ["https://ai.azure.com/.default"]}
    print("auth: DefaultAzureCredential")

client = FineTuningSessionClient(endpoint=PROJECT_ENDPOINT, credential=credential, **extra)

# Convenience: every management call needs these two kwargs.
OPTS = dict(foundry_features=PREVIEW, api_version=API_VERSION)

## 3. List sessions

Pulls one page at a time. Default `limit` is service-decided; pass an explicit `limit` to bound output.

In [ ]:
page = client.sessions.list(limit=50, offset=0, **OPTS)

print(f"{len(page.data)} of {page.cursor.total_count} sessions (offset {page.cursor.offset}, limit {page.cursor.limit})\n")
header = f"{'session_id':<24} {'status':<10} {'base_model':<28} {'rank':>4}  last_request_time"
print(header)
print("-" * len(header))
for s in page.data:
    status = s.status.value if hasattr(s.status, "value") else s.status
    print(f"{s.session_id:<24} {status:<10} {s.base_model:<28} {str(s.lora_rank or '-'):>4}  {s.last_request_time}")

### Page through everything

If you have more sessions than `limit`, this fetches all pages and stuffs them in `all_sessions` for filtering below.

In [ ]:
all_sessions = []
offset = 0
limit = 100
while True:
    page = client.sessions.list(limit=limit, offset=offset, **OPTS)
    all_sessions.extend(page.data)
    offset += len(page.data)
    if offset >= page.cursor.total_count or not page.data:
        break
print(f"fetched {len(all_sessions)} sessions")

### Aggregate view

Counts by status + base model, plus a few “interesting” slices: anything currently `ready` (holding GPU), anything `corrupted` (state lost, can’t resume), anything `failed`, and the most-recently-active session per base model. Pure local filtering over `all_sessions` — no extra service calls.

In [ ]:
from collections import Counter, defaultdict
import datetime

def _status(s):
    return s.status.value if hasattr(s.status, "value") else s.status

# ── Counts by status ──────────────────────────────────────────────────
by_status = Counter(_status(s) for s in all_sessions)
print("by status:")
for k, v in sorted(by_status.items(), key=lambda kv: -kv[1]):
    print(f"  {k:<10} {v:>4}")

# ── Counts by base model ──────────────────────────────────────────────
by_model = Counter(s.base_model for s in all_sessions)
print("\nby base_model:")
for k, v in sorted(by_model.items(), key=lambda kv: -kv[1]):
    print(f"  {k:<32} {v:>4}")

# ── Cross-tab: status × base_model ────────────────────────────────────
cross = defaultdict(Counter)
for s in all_sessions:
    cross[s.base_model][_status(s)] += 1
statuses = sorted({_status(s) for s in all_sessions})
print(f"\nstatus x base_model:")
print(f"  {'base_model':<32} " + "  ".join(f"{st:>9}" for st in statuses))
for model, row in sorted(cross.items(), key=lambda kv: -sum(kv[1].values())):
    print(f"  {model:<32} " + "  ".join(f"{row.get(st, 0):>9}" for st in statuses))


In [ ]:
# ── Interesting slices ────────────────────────────────────────────────
def _show(label, rows, *, limit=10):
    print(f"\n{label} ({len(rows)}):")
    if not rows:
        print("  — none —")
        return
    for s in rows[:limit]:
        print(f"  {s.session_id:<24} {_status(s):<10} {s.base_model:<28} "
              f"rank={str(s.lora_rank or '-'):>3}  last={s.last_request_time}")
    if len(rows) > limit:
        print(f"  … and {len(rows) - limit} more")

now = datetime.datetime.now(datetime.timezone.utc)

ready    = [s for s in all_sessions if _status(s) == "ready"]
failed   = [s for s in all_sessions if _status(s) == "failed"]
corrupted = [s for s in all_sessions if getattr(s, "corrupted", False)]
recent   = sorted(all_sessions, key=lambda s: s.last_request_time, reverse=True)

_show("currently ready (holding GPU)", sorted(ready, key=lambda s: s.last_request_time, reverse=True))
_show("failed",                              sorted(failed, key=lambda s: s.last_request_time, reverse=True))
_show("corrupted (state lost, not resumable)", corrupted)
_show("most recently active",                 recent, limit=5)

# Most-recently-active session *per base model* — useful for picking a continual-FT source.
latest_per_model = {}
for s in recent:  # already sorted desc
    latest_per_model.setdefault(s.base_model, s)
print(f"\nmost recently active per base_model ({len(latest_per_model)}):")
for model, s in sorted(latest_per_model.items()):
    print(f"  {model:<32} {s.session_id:<24} {_status(s):<10} last={s.last_request_time}")

## 4. Inspect a single session

Set `SESSION_ID` to one of the IDs printed above (or copy it from a `run_meta.json` under `~/interactive-post-training-runs/<recipe>/<run>/run_meta.json` &rarr; `azure.session_id`).

In [ ]:
# Default to the most recently active session, or assign a known session ID.
if not all_sessions:
    raise ValueError("No sessions were returned. Create a session first or set SESSION_ID to a known session ID.")
SESSION_ID = max(all_sessions, key=lambda item: item.last_request_time).session_id
print("inspecting:", SESSION_ID)


def _enum(value):
    return value.value if hasattr(value, "value") else value


s = client.sessions.get(session_id=SESSION_ID, **OPTS)
print(f"  session_id  : {s.session_id}")
print(f"  type        : {_enum(s.type)}")
print(f"  status      : {_enum(s.status)}")
print(f"  base_model  : {s.model_data.base_model}")
print(f"  model_name  : {s.model_data.model_name}")
print(f"  lora_config : {s.model_data.lora_config}")

## 5. List & inspect checkpoints

`checkpoint_type` is `training` (full state — what you pass as `load_checkpoint_path` for continual-FT) or `sampler` (LoRA weights snapshots used between steps to refresh the sampler).

In [ ]:
from collections import Counter

ckpts = client.checkpoints.list(session_id=SESSION_ID, **OPTS)
by_type = Counter(_enum(c.checkpoint_type) for c in ckpts.checkpoints)
summary = ", ".join(f"{n} {t}" for t, n in by_type.most_common()) or "none"
print(f"{len(ckpts.checkpoints)} checkpoints for {SESSION_ID}  ({summary})\n")
for c in sorted(ckpts.checkpoints, key=lambda c: c.time):
    print(f"  {c.checkpoint_id:<16} {_enum(c.checkpoint_type):<10} {c.time}")

In [ ]:
training_ckpts = [c for c in ckpts.checkpoints if _enum(c.checkpoint_type) == "training"]
if training_ckpts:
    latest_training = max(training_ckpts, key=lambda checkpoint: checkpoint.time)
    info = client.checkpoints.get(
        session_id=SESSION_ID,
        checkpoint_id=latest_training.checkpoint_id,
        **OPTS,
    )
    print(f"latest training checkpoint of {SESSION_ID}:")
    print(f"  base_model : {info.base_model}")
    print(f"  is_lora    : {info.is_lora}")
    print(f"  lora_rank  : {info.lora_rank}")
else:
    print("no training checkpoints yet — only sampler snapshots")

## 6. Close local client resources

Run the next cell when finished, including after an earlier cell fails. This closes local HTTP/credential resources only; it does not unload or delete remote sessions.

In [ ]:
try:
    client.close()
finally:
    if not api_key:
        credential.close()